In [3]:
import os
import re

In [4]:
docs_md_path = os.path.abspath("../documents/processed")
docs_md_cats  =os.listdir(docs_md_path)
print(f"Categories: {docs_md_cats}")

Categories: ['workout', 'nutrition']


In [5]:
import re


def split_on_page_markers(text: str) -> list[str]:
    """
    Split on '# Page N' markers and return page contents.
    Keeps content order and ignores empty pages.
    """
    # Split on '# Page <number>' markers
    parts = re.split(r'^\s*#\s*Page\s*\d+\s*$', text, flags=re.MULTILINE)
    pages = [p.strip() for p in parts if p and p.strip()]
    return pages


def remove_page_markers(text: str) -> str:
    # Remove residual 'Page N' lines and standalone page numbers
    text = re.sub(r'Page\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    return text


def remove_pdf_noise(text: str) -> str:
    noise_patterns = [
        r"Downloaded from.*",
        r"Unauthorized reproduction.*",
        r"Copyright.*",
        r"DOI:\s*\S+",
        r"Author Manuscript.*",
        r"HHS Public Access.*",
    ]
    for pattern in noise_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    text = re.sub(r"\n.*\|\s*DOI:.*\n", "\n", text)
    text = re.sub(r"Page\s*\d+(\s*of\s*\d+)?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)
    text = re.sub(r"\n[-–—]*\s*\d+\s*[-–—]*\n", "\n", text)
    text = re.sub(r"-\n", "", text)
    return text


def remove_journal_headers(text: str) -> str:
    return re.sub(
        r'^.*(?:et al|Br J|doi:10\.\d+).*$\n?',
        '',
        text,
        flags=re.MULTILINE | re.IGNORECASE
    )


def remove_author_blocks(text: str) -> str:
    return re.sub(
        r'^[A-Z][a-z]+ [A-Z]\.?\s+[A-Z][a-z]+,?\s*\d+(,?\s*[A-Z][a-z]+ [A-Z]\.?\s+[A-Z][a-z]+,?\s*\d+)+\s*$',
        '',
        text,
        flags=re.MULTILINE
    )


def remove_affiliations(text: str) -> str:
    text = re.sub(
        r'^\d+\s+(.*(?:University|Institute|Department|School|Center|Faculty).*)$',
        '',
        text,
        flags=re.MULTILINE | re.IGNORECASE
    )
    text = re.sub(
        r'^[A-Z][a-z]+,\s+[A-Z]{2}\s+\d{5}.*$',
        '',
        text,
        flags=re.MULTILINE
    )
    return text


def remove_correspondence(text: str) -> str:
    text = re.sub(
        r'(?:Correspondence|Corresponding author|Address|Reprint)[^\n]*(?:\n[^\n]*)?',
        '',
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(r'(Phone|Fax|Tel)[:.]?\s*[\d\s\-\(\)]+', '', text, flags=re.IGNORECASE)
    return text


def remove_urls_and_emails(text: str) -> str:
    text = re.sub(r'https?://\S+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'www\.\S+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[a-z]{3,}\s+\.\s+[a-z]{3,}\s+\.\s+[a-z]{3,}', '', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
    return text


def remove_unnecessary_sections(text: str) -> str:
    # Remove abstracts/summaries
    text = re.sub(
        r'^(?:ABSTRACT|Abs\s*Tr\s*ACT|SUMMARY).*?(?=^(?:INTRODUCTION|Background|CHAPTER|METHODS|Results|In[^a-z]))',
        '',
        text,
        flags=re.MULTILINE | re.IGNORECASE | re.DOTALL
    )
    # Remove keywords
    text = re.sub(
        r'^(?:KEYWORDS|Key Words).*?(?=^(?:INTRODUCTION|Background|CHAPTER|METHODS))',
        '',
        text,
        flags=re.MULTILINE | re.IGNORECASE | re.DOTALL
    )
    # Remove references/bibliography sections
    text = re.sub(
        r'^(?:References|REFERENCES|Bibliography|BIBLIOGRAPHY)\b.*',
        '',
        text,
        flags=re.MULTILINE | re.IGNORECASE | re.DOTALL
    )
    # Remove numbered reference lists
    text = re.sub(
        r'(\n\s*\d+\.\s+[A-Z][a-z]+.*){3,}.*',
        '',
        text,
        flags=re.DOTALL
    )
    # Remove Table of Contents blocks
    text = re.sub(
        r'^Table of Contents.*?(?=^(?:CHAPTER|Introduction|CHAPTER ONE|CHAPTER TWO))',
        '',
        text,
        flags=re.MULTILINE | re.IGNORECASE | re.DOTALL
    )
    return text


def merge_lines(text: str) -> str:
    return re.sub(r'(?<!\n)\n(?!\n)', ' ', text)


def normalize_spaces(text: str) -> str:
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'\s+([.,;:!?])', r'\1', text)
    return text


def normalize_empty_lines(text: str) -> str:
    text = '\n'.join(line.rstrip() for line in text.split('\n'))
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text


def remove_orphan_lines(text: str) -> str:
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped and not re.match(r'^[-–—\d\s]+$', stripped):
            cleaned_lines.append(line)
    return '\n'.join(cleaned_lines)


def clean_page(text: str) -> str:
    text = remove_page_markers(text)
    text = remove_pdf_noise(text)
    text = remove_journal_headers(text)
    text = remove_author_blocks(text)
    text = remove_affiliations(text)
    text = remove_correspondence(text)
    text = remove_urls_and_emails(text)
    text = remove_unnecessary_sections(text)
    text = merge_lines(text)
    text = normalize_spaces(text)
    text = normalize_empty_lines(text)
    text = remove_orphan_lines(text)
    return text.strip()


def clean_text(text: str, min_words_per_page: int = 50) -> str:
    pages = split_on_page_markers(text)
    cleaned_pages = []

    for page in pages:
        cleaned = clean_page(page)
        if cleaned and len(cleaned.split()) >= min_words_per_page:
            cleaned_pages.append(cleaned)

    return "\n\n#\n\n".join(cleaned_pages).strip()

In [6]:
docs_md_cleaned_path = os.path.abspath("../documents/cleaned")
os.makedirs(docs_md_cleaned_path, exist_ok=True)

for cat in docs_md_cats:
    cat_path = os.path.join(docs_md_path, cat)
    cat_cleaned_path = os.path.join(docs_md_cleaned_path, cat)
    os.makedirs(cat_cleaned_path, exist_ok=True)
    for doc in os.listdir(cat_path):
        doc_path = os.path.join(cat_path, doc)
        with open(doc_path, "r", encoding="utf-8") as f:
            text = f.read()
        cleaned_text = clean_text(text)
        cleaned_doc_path = os.path.join(cat_cleaned_path, doc)
        with open(cleaned_doc_path, "w", encoding="utf-8") as f:
            f.write(cleaned_text)
            print(f"doc : {doc} cleaned and saved.")
            print(f"Original length: {len(text)}, Cleaned length: {len(cleaned_text)}")

doc : the_mechanisms_of_muscle_hypertrophy_and_their.40.md cleaned and saved.
Original length: 103192, Cleaned length: 60075
doc : progression_models_in_resistance_training_for.26.md cleaned and saved.
Original length: 140643, Cleaned length: 88830
doc : basics_of_strength_and_conditioning_manual.md cleaned and saved.
Original length: 230257, Cleaned length: 94028
doc : mss-51-094.md cleaned and saved.
Original length: 53412, Cleaned length: 44846
doc : nihms851543.md cleaned and saved.
Original length: 51629, Cleaned length: 31468
doc : 12970_2017_Article_189.md cleaned and saved.
Original length: 132182, Cleaned length: 87225
doc : ncomms13103.md cleaned and saved.
Original length: 67169, Cleaned length: 54944
doc : 439.full.md cleaned and saved.
Original length: 113350, Cleaned length: 91098
